# Phase E Validation: Temporal Resolution and Discretization Limits

This notebook runs empirical sweeps to bound the parameters of the microscopic simulation:

1. **`seconds_per_tick` (Exp 1)**: At what granularity does discretization error spike? We sweep geometrically from 10s to 320s, keeping simulated duration fixed at 30 min.
2. **`num_ticks` (Exp 2)**: What is the *minimum* simulation horizon for stable, converged fitness? We use bisection per (route scale, optimal spt) targeting CV ≤ 5%.
3. **2D Independence Check (Exp 3)**: A spot-check heatmap confirming that `spt` and `sim_minutes` effects are approximately separable — justifying the sequential optimization strategy above.

**Route scales tested**: 3, 6, 9, 12 routes (with proportionally scaled jeep counts: 30, 60, 90, 120).

**Stability criterion**: We adopt a **CV ≤ 5%** threshold throughout, following the relative precision standard established in discrete-event simulation output analysis literature. Banks et al. (2010) define relative precision as the ratio of the confidence interval half-width to the sample mean, and recommend ≤ 5% as the accepted threshold for terminating replications — a criterion directly equivalent to bounding the coefficient of variation at this level (Banks, J., Carson, J. S., Nelson, B. L., & Nicol, D. M. (2010). *Discrete-Event System Simulation*, 5th ed. Prentice Hall).

**Hypothesis**: `spt` bounds are physics-limited and largely scale-invariant; `num_ticks` bounds scale upward with system complexity as larger systems take longer to reach steady state.

In [ ]:
import os
import sys
import random
import time
import yaml
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from itertools import product as iproduct

random.seed(42)
np.random.seed(42)

sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

from utils_simplified import (
    reuse_citygraph, reuse_ddm, generate_route_system,
    generate_dummy_yaml, build_travelgraph, run_simulation
)
from utils.jeep import Jeep
from utils.jeep_system import JeepSystem

cg = reuse_citygraph("results_and_discussion/pkl/profile_p1.pkl")
ddm = reuse_ddm("results_and_discussion/pkl/ddm_8am.pkl")
os.makedirs("configs", exist_ok=True)
os.makedirs("results_and_discussion/pkl", exist_ok=True)

## 0. Experimental Configuration

All tunable knobs are declared here. The jeep count scales linearly with route count (10 jeeps/route), matching the vehicle density used in the main optimization pipeline. Spawn rate is held constant at 120/hr across all scales so that per-vehicle load scales realistically.

The **CV ≤ 5% stability threshold** (`CV_TARGET`) is the operationalization of the relative precision criterion from Banks et al. (2010). Relative precision is defined as the confidence interval half-width divided by the sample mean; for a t-distribution with moderate n this is approximately equivalent to CV × (t / √n), so bounding CV at 5% ensures relative precision well within the 5% threshold across all tested rep counts.

In [ ]:
# --- Route scales ---
ROUTE_COUNTS    = [3, 6, 9, 12]
JEEPS_PER_ROUTE = 10          # → 30, 60, 90, 120 jeeps respectively
SPAWN_RATE      = 120.0       # passengers/hr, fixed across all scales

# --- Exp 1: seconds_per_tick sweep ---
# Geometric spacing: uniform sensitivity per decade.
# [10, 20, 40, 80, 160, 320] gives 6 points across 1.5 decades.
SPT_VALUES       = [10, 20, 40, 80, 160, 320]
FIXED_SIM_S_EXP1 = 1800        # 30 simulated minutes — enough for steady-state at spt=10
N_REPS_EXP1      = 8           # reps per (route_count, spt) cell

# --- Exp 2: num_ticks bisection ---
# CV threshold: 5% relative precision, per Banks et al. (2010) Discrete-Event System
# Simulation (5th ed.), the standard DES stopping criterion for terminating simulations.
CV_TARGET        = 0.05        # stability threshold: CV ≤ 5%
BISECT_BOUNDS    = (600, 7200) # simulated seconds: 10 min → 2 hr
BISECT_ITERS     = 6           # bisection depth: resolves to ±(7200-600)/2^6 ≈ ±100s
N_REPS_EXP2      = 8           # reps per bisection probe

# --- Exp 3: 2D spot-check ---
# Coarser grid — existence check only, not exhaustive.
SPT_2D   = [10, 40, 160]       # representative low / mid / high
SIM_S_2D = [900, 1800, 3600, 5400]  # 15, 30, 60, 90 min
N_REPS_EXP3 = 5
ROUTES_2D = 6                  # single scale for the heatmap

print("Configuration summary:")
for n in ROUTE_COUNTS:
    print(f"  {n} routes → {n * JEEPS_PER_ROUTE} jeeps")
print(f"  Exp 1 spt values: {SPT_VALUES}")
print(f"  Exp 2 bisection range: {BISECT_BOUNDS[0]}–{BISECT_BOUNDS[1]}s, {BISECT_ITERS} iters")
print(f"  CV stability target: {CV_TARGET:.0%} (relative precision criterion, Banks et al. 2010)")

## 1. Topographic Initialization

Route systems are pre-generated once per scale with fixed seeds to eliminate topology variation as a confound. Jeep count scales with route count; each jeep starts equidistantly distributed along its assigned route.

In [ ]:
print("[SETUP] Generating route systems...")
route_systems = {}
for n_routes in ROUTE_COUNTS:
    random.seed(42 + n_routes)
    np.random.seed(42 + n_routes)
    route_systems[n_routes] = generate_route_system(n_routes, cg, ddm)
    total_edges = sum(len(r.path) for r in route_systems[n_routes])
    print(f"  {n_routes} routes | {total_edges} total edges | {n_routes * JEEPS_PER_ROUTE} jeeps")


def make_jeep_system(routes, spt):
    """Build a JeepSystem with JEEPS_PER_ROUTE jeeps per route."""
    jeeps = []
    n_jeeps = len(routes) * JEEPS_PER_ROUTE
    per_route = max(1, n_jeeps // len(routes))
    for r in routes:
        for _ in range(per_route):
            start = (r.path[0].start.lon, r.path[0].start.lat)
            jeeps.append(Jeep(r, curr_pos=start, speed=20.0,
                               max_capacity=16, seconds_per_tick=spt))
    return JeepSystem(jeeps=jeeps, routes=routes,
                      weight_tolerance=50.0, equidistant_spawn=True)


def run_one(routes, spt, num_ticks, rep_seed, label="tmp"):
    """Run a single microscopic simulation. Returns (fitness, metrics_dict)."""
    yaml_path = f"configs/_nb5_{label}_{spt}_{num_ticks}_{rep_seed}.yaml"
    n_jeeps = len(routes) * JEEPS_PER_ROUTE
    generate_dummy_yaml(
        yaml_path,
        **{
            "simulation.num_ticks":              num_ticks,
            "simulation.seconds_per_tick":        spt,
            "simulation.total_allocatable_jeeps": n_jeeps,
            "simulation.spawn_rate_per_hour":     SPAWN_RATE,
            "simulation.mohring_sample_size":     2,
        }
    )
    random.seed(rep_seed)
    np.random.seed(rep_seed)
    js = make_jeep_system(routes, spt)
    tg = build_travelgraph(cg, yaml_path, routes)
    sim = run_simulation(tg, yaml_path, js, ddm, delete_yaml_when_done=True)
    res = sim.evaluate_fitness()
    return res.fitness_score, res.metrics


def compute_cv(fitness_list):
    """Coefficient of variation; returns 0 if mean is near zero."""
    arr = np.array(fitness_list)
    m = arr.mean()
    return (arr.std() / m) if m > 1e-9 else 0.0

---
## 2. Experiment 1 — Temporal Discretization Sensitivity (`seconds_per_tick`)

**Design**: Sweep `spt` ∈ {10, 20, 40, 80, 160, 320} while holding simulated duration fixed at 1800s (30 min). `num_ticks = 1800 / spt`. This isolates the discretization effect: every condition represents the *same physical scenario* at different temporal resolutions.

**Expected finding**: CV and mean fitness are stable across the low-spt plateau, then rise sharply at the cliff where tick granularity becomes too coarse for vehicles to traverse short edges or for passengers to board/alight within a single tick. The cliff location may shift slightly with route scale (denser routes have shorter edges on average).

**N = 8 reps** per cell → sufficient to estimate CV reliably (SE of CV ≈ CV/√(2N-2) ≈ 0.25 CV).

In [ ]:
spt_rows = []
total_cells = len(ROUTE_COUNTS) * len(SPT_VALUES)
cell_i = 0

for n_routes in ROUTE_COUNTS:
    routes = route_systems[n_routes]
    for spt in SPT_VALUES:
        cell_i += 1
        num_ticks = max(10, FIXED_SIM_S_EXP1 // spt)
        fits_this_cell = []
        print(f"\n[Exp1 {cell_i}/{total_cells}] R={n_routes} | spt={spt:3d}s | ticks={num_ticks:4d}")

        for rep in range(N_REPS_EXP1):
            rep_seed = 1000 + n_routes * 100 + spt * 10 + rep
            t0 = time.time()
            fitness, metrics = run_one(routes, spt, num_ticks, rep_seed, label="e1")
            wall = time.time() - t0
            fits_this_cell.append(fitness)

            spt_rows.append({
                "n_routes":     n_routes,
                "spt":          spt,
                "num_ticks":    num_ticks,
                "rep":          rep,
                "fitness":      fitness,
                "completed":    metrics.get("completed_count",  0),
                "incomplete":   metrics.get("incomplete_count", 0),
                "mean_commute": metrics.get("mean_commute_time", 0),
                "wall_s":       wall,
            })
            print(f"  rep {rep+1}/{N_REPS_EXP1} | fit={fitness:10.2f} "
                  f"done={spt_rows[-1]['completed']:3d} "
                  f"inc={spt_rows[-1]['incomplete']:3d} "
                  f"wall={wall:4.1f}s")
            gc.collect()

        running_cv = compute_cv(fits_this_cell)
        print(f"  → cell CV={running_cv:.4f}")

df_spt = pd.DataFrame(spt_rows)

In [ ]:
agg_spt = df_spt.groupby(["n_routes", "spt"]).agg(
    mean_fit     =("fitness",  "mean"),
    std_fit      =("fitness",  "std"),
    mean_wall    =("wall_s",   "mean"),
    mean_done    =("completed","mean"),
    mean_commute =("mean_commute", "mean"),
).reset_index()
agg_spt["cv_fit"] = agg_spt["std_fit"] / agg_spt["mean_fit"]

# Identify cliff per route scale: first spt where CV exceeds CV_TARGET
cliff_spt = {}
for n in ROUTE_COUNTS:
    sub = agg_spt[agg_spt.n_routes == n].sort_values("spt")
    over = sub[sub.cv_fit > CV_TARGET]
    cliff_spt[n] = int(over.iloc[0]["spt"]) if len(over) else SPT_VALUES[-1]
    print(f"  R={n}: CV cliff at spt={cliff_spt[n]}s (CV={sub[sub.spt==cliff_spt[n]].cv_fit.values[0]:.4f})")

# Conservative anchor: tightest cliff across all scales
OPTIMAL_SPT = min(cliff_spt.values()) // 2  # one step below cliff
OPTIMAL_SPT = max(OPTIMAL_SPT, SPT_VALUES[0])
print(f"\n★ Anchoring seconds_per_tick = {OPTIMAL_SPT}s for Exp 2")

In [ ]:
palette_routes = sns.color_palette("tab10", n_colors=len(ROUTE_COUNTS))
route_color = {n: palette_routes[i] for i, n in enumerate(ROUTE_COUNTS)}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
ax_cv, ax_wall, ax_fit, ax_done = axes.flat

# 1. CV vs spt
for n in ROUTE_COUNTS:
    sub = agg_spt[agg_spt.n_routes == n].sort_values("spt")
    ax_cv.plot(sub.spt, sub.cv_fit, marker="o", linewidth=2.2,
               color=route_color[n], label=f"{n} routes")
ax_cv.axhline(CV_TARGET, color="red", linestyle="--", linewidth=1.4,
              label=f"CV threshold ({CV_TARGET:.0%}, Banks et al. 2010)")
ax_cv.set_title("Fitness Consistency vs Discretization Step",
                fontsize=12, fontweight="bold")
ax_cv.set_xlabel("seconds_per_tick", fontsize=10)
ax_cv.set_ylabel("Coefficient of Variation (CV)", fontsize=10)
ax_cv.set_xscale("log", base=2)
ax_cv.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax_cv.legend(fontsize=9)

# 2. Wall time vs spt
for n in ROUTE_COUNTS:
    sub = agg_spt[agg_spt.n_routes == n].sort_values("spt")
    ax_wall.plot(sub.spt, sub.mean_wall, marker="s", linewidth=2.2,
                 color=route_color[n], label=f"{n} routes")
ax_wall.set_title("Execution Wall-Time vs Discretization Step",
                  fontsize=12, fontweight="bold")
ax_wall.set_xlabel("seconds_per_tick", fontsize=10)
ax_wall.set_ylabel("Mean Runtime (s)", fontsize=10)
ax_wall.set_xscale("log", base=2)
ax_wall.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax_wall.invert_xaxis()
ax_wall.legend(fontsize=9)

# 3. Mean fitness vs spt
for n in ROUTE_COUNTS:
    sub = agg_spt[agg_spt.n_routes == n].sort_values("spt")
    ax_fit.errorbar(sub.spt, sub.mean_fit, yerr=sub.std_fit,
                    marker="D", linewidth=2.0, capsize=4,
                    color=route_color[n], label=f"{n} routes")
ax_fit.set_title("Mean Fitness (±1 SD) vs Discretization Step",
                 fontsize=12, fontweight="bold")
ax_fit.set_xlabel("seconds_per_tick", fontsize=10)
ax_fit.set_ylabel("Mean Fitness Score", fontsize=10)
ax_fit.set_xscale("log", base=2)
ax_fit.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax_fit.legend(fontsize=9)

# 4. Mean commute time vs spt (bias indicator)
for n in ROUTE_COUNTS:
    sub = agg_spt[agg_spt.n_routes == n].sort_values("spt")
    ax_done.plot(sub.spt, sub.mean_commute, marker="^", linewidth=2.2,
                 color=route_color[n], label=f"{n} routes")
ax_done.set_title("Mean Commute Time vs Discretization Step",
                  fontsize=12, fontweight="bold")
ax_done.set_xlabel("seconds_per_tick", fontsize=10)
ax_done.set_ylabel("Mean Commute Time (s)", fontsize=10)
ax_done.set_xscale("log", base=2)
ax_done.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax_done.legend(fontsize=9)

fig.suptitle("Visual 1: Temporal Discretization Limits (Exp 1)\n"
             f"Fixed simulated duration = {FIXED_SIM_S_EXP1}s, N={N_REPS_EXP1} reps",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("results_and_discussion/pkl/visual1_spt_sweep.pdf",
            bbox_inches="tight")
plt.show()

---
## 3. Experiment 2 — Minimum Simulation Duration via Bisection (`num_ticks`)

**Design**: With `spt` anchored at `OPTIMAL_SPT` (from Exp 1), we bisect over simulated-second values in [`BISECT_BOUNDS`] to find the minimum duration at which CV ≤ 5%. This is well-suited for bisection because variance collapse is monotone in simulation length.

**Stopping criterion**: The 5% CV threshold follows the relative precision standard in Banks et al. (2010, *Discrete-Event System Simulation*, 5th ed.), the canonical DES output analysis reference. Relative precision is defined as the CI half-width / sample mean ≤ 5%, which CV at this level satisfies.

**Bisection protocol**: At each iteration, probe the midpoint. If CV ≥ threshold, raise the lower bound; otherwise, lower the upper bound. After `BISECT_ITERS` iterations, record the last satisfying point.

In [ ]:
bisect_log   = []  # full probe history
bisect_final = {}  # n_routes → min stable sim_seconds

for n_routes in ROUTE_COUNTS:
    routes = route_systems[n_routes]
    lo, hi = BISECT_BOUNDS
    last_stable_s = hi  # conservative fallback

    print(f"\n{'='*60}")
    print(f"[Exp2] Bisection for R={n_routes} | spt={OPTIMAL_SPT}")
    print(f"  Search range: {lo}–{hi}s ({lo//60}–{hi//60} min)")
    print(f"{'='*60}")

    for bisect_iter in range(BISECT_ITERS):
        mid_s    = (lo + hi) // 2
        # Round to a multiple of spt so num_ticks is an integer
        mid_s    = max(OPTIMAL_SPT, (mid_s // OPTIMAL_SPT) * OPTIMAL_SPT)
        num_ticks = mid_s // OPTIMAL_SPT

        fits = []
        for rep in range(N_REPS_EXP2):
            rep_seed = 3000 + n_routes * 1000 + mid_s + rep
            fitness, metrics = run_one(routes, OPTIMAL_SPT, num_ticks,
                                       rep_seed, label="e2")
            fits.append(fitness)
            gc.collect()

        cv   = compute_cv(fits)
        stable = cv < CV_TARGET
        bisect_log.append({
            "n_routes":    n_routes,
            "bisect_iter": bisect_iter,
            "sim_s":       mid_s,
            "sim_min":     round(mid_s / 60, 1),
            "num_ticks":   num_ticks,
            "mean_fit":    np.mean(fits),
            "cv_fit":      cv,
            "stable":      stable,
        })

        flag = "✓ stable" if stable else "✗ noisy"
        print(f"  iter {bisect_iter+1}/{BISECT_ITERS}: sim={mid_s:5d}s "
              f"({mid_s//60}min) | CV={cv:.4f} [{flag}]")

        if stable:
            last_stable_s = mid_s
            hi = mid_s   # search lower
        else:
            lo = mid_s   # need more time

    bisect_final[n_routes] = last_stable_s
    print(f"  → Minimum stable duration for R={n_routes}: {last_stable_s}s "
          f"({last_stable_s//60} min)")

df_bisect = pd.DataFrame(bisect_log)
print("\n[Exp2 Summary]")
for n, s in bisect_final.items():
    print(f"  R={n:2d}: min stable sim = {s:5d}s ({s//60} min)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. Bisection trajectory: CV vs sim_min per route scale
for n in ROUTE_COUNTS:
    sub = df_bisect[df_bisect.n_routes == n].sort_values("sim_min")
    ax1.plot(sub.sim_min, sub.cv_fit, marker="o", linewidth=2.0,
             color=route_color[n], label=f"{n} routes",
             alpha=0.85)
    # Mark the bisection-selected minimum
    min_s = bisect_final[n]
    row = df_bisect[(df_bisect.n_routes == n) & (df_bisect.sim_s == min_s)]
    if len(row):
        ax1.axvline(min_s / 60, color=route_color[n],
                    linestyle=":", linewidth=1.2, alpha=0.7)

ax1.axhline(CV_TARGET, color="red", linestyle="--", linewidth=1.4,
            label=f"CV threshold ({CV_TARGET:.0%}, Banks et al. 2010)")
ax1.set_title("Variance Collapse: Bisection Probes",
              fontsize=12, fontweight="bold")
ax1.set_xlabel("Simulated Duration (min)", fontsize=10)
ax1.set_ylabel("CV of Fitness", fontsize=10)
ax1.legend(fontsize=9)

# 2. Minimum stable duration vs route count (scaling law)
ns     = sorted(bisect_final.keys())
min_ss = [bisect_final[n] / 60 for n in ns]
ax2.bar([str(n) for n in ns], min_ss,
        color=[route_color[n] for n in ns], edgecolor="white",
        linewidth=0.8)
for i, (n, v) in enumerate(zip(ns, min_ss)):
    ax2.text(i, v + 0.5, f"{v:.0f} min",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.set_title("Minimum Stable Duration vs System Scale",
              fontsize=12, fontweight="bold")
ax2.set_xlabel("Number of Routes", fontsize=10)
ax2.set_ylabel("Minimum Stable Simulated Duration (min)", fontsize=10)

fig.suptitle(f"Visual 2: Simulation Duration Bisection (Exp 2)\n"
             f"spt={OPTIMAL_SPT}s, N={N_REPS_EXP2} reps/probe, "
             f"{BISECT_ITERS} bisection iters",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("results_and_discussion/pkl/visual2_ticks_bisection.pdf",
            bbox_inches="tight")
plt.show()

---
## 4. Experiment 3 — 2D Independence Check: `spt` × Simulated Duration

**Purpose**: The sequential strategy in Exp 1–2 only makes sense if the two parameters are approximately separable — i.e., the optimal `spt` doesn't depend on simulation length and vice versa. This experiment checks that assumption with a small 3×4 heatmap grid at a single representative route scale (6 routes).

**If the heatmap shows strong interaction effects** (e.g., a diagonal structure where CV is only low when both parameters are simultaneously within a narrow band), then a joint 2D search would be warranted. If the heatmap shows approximately horizontal or vertical iso-CV contours, the sequential approach is validated.

**N = 5 reps** per cell is lighter here — sufficient to confirm the existence/absence of interaction, not to characterize the exact boundaries.

In [ ]:
routes_2d = route_systems[ROUTES_2D]
heatmap_rows = []
total_2d = len(SPT_2D) * len(SIM_S_2D)
cell_2d = 0

for spt, sim_s in iproduct(SPT_2D, SIM_S_2D):
    cell_2d += 1
    num_ticks = max(10, sim_s // spt)
    fits = []
    print(f"[Exp3 {cell_2d}/{total_2d}] spt={spt:3d}s | sim={sim_s//60:2d}min "
          f"| ticks={num_ticks:4d}", end="  ", flush=True)

    for rep in range(N_REPS_EXP3):
        rep_seed = 4000 + spt * 100 + sim_s + rep
        fitness, metrics = run_one(routes_2d, spt, num_ticks,
                                   rep_seed, label="e3")
        fits.append(fitness)
        gc.collect()

    cv = compute_cv(fits)
    heatmap_rows.append({
        "spt": spt, "sim_s": sim_s, "sim_min": sim_s // 60,
        "num_ticks": num_ticks,
        "mean_fit": np.mean(fits), "cv_fit": cv,
    })
    print(f"CV={cv:.4f}")

df_2d = pd.DataFrame(heatmap_rows)

In [ ]:
pivot_cv   = df_2d.pivot(index="spt",     columns="sim_min", values="cv_fit")
pivot_fit  = df_2d.pivot(index="spt",     columns="sim_min", values="mean_fit")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# CV heatmap
sns.heatmap(pivot_cv, annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=0.5, ax=ax1,
            cbar_kws={"label": "CV of Fitness"})
ax1.set_title(f"CV Heatmap: spt × sim_duration\n"
              f"(R={ROUTES_2D} routes, N={N_REPS_EXP3} reps)",
              fontsize=12, fontweight="bold")
ax1.set_xlabel("Simulated Duration (min)", fontsize=10)
ax1.set_ylabel("seconds_per_tick", fontsize=10)

# Mean fitness heatmap
sns.heatmap(pivot_fit, annot=True, fmt=".1f", cmap="viridis",
            linewidths=0.5, ax=ax2,
            cbar_kws={"label": "Mean Fitness"})
ax2.set_title(f"Mean Fitness Heatmap: spt × sim_duration\n"
              f"(R={ROUTES_2D} routes, N={N_REPS_EXP3} reps)",
              fontsize=12, fontweight="bold")
ax2.set_xlabel("Simulated Duration (min)", fontsize=10)
ax2.set_ylabel("seconds_per_tick", fontsize=10)

fig.suptitle("Visual 3: 2D Independence Check (Exp 3)\n"
             "Approximately vertical iso-CV contours → parameters are separable",
             fontsize=13, fontweight="bold", y=1.04)
plt.tight_layout()
plt.savefig("results_and_discussion/pkl/visual3_2d_heatmap.pdf",
            bbox_inches="tight")
plt.show()

---
## 5. Summary and Recommended Hyperparameters

This cell consolidates all findings and prints the recommended simulation configuration for the main optimization pipeline.

In [ ]:
print("=" * 60)
print("PHASE E VALIDATION SUMMARY")
print("=" * 60)

print("\n[Exp 1] seconds_per_tick cliff points:")
for n, s in cliff_spt.items():
    row = agg_spt[(agg_spt.n_routes == n) & (agg_spt.spt == s)]
    cv_val = row.cv_fit.values[0] if len(row) else float("nan")
    print(f"  R={n:2d}: cliff at spt={s:3d}s  (CV={cv_val:.4f})")
print(f"  → Anchored spt = {OPTIMAL_SPT}s (half of tightest cliff)")

print("\n[Exp 2] Minimum stable simulation durations:")
for n, s in bisect_final.items():
    print(f"  R={n:2d}: {s:5d}s ({s//60} min)")

print("\n[Exp 3] 2D Interaction:")
# Compute row-wise CV range to detect interaction
row_ranges = pivot_cv.apply(lambda row: row.max() - row.min(), axis=1)
col_ranges = pivot_cv.apply(lambda col: col.max() - col.min(), axis=0)
print(f"  CV range across sim_min (per spt level):   {row_ranges.mean():.4f} avg")
print(f"  CV range across spt (per sim_min level):   {col_ranges.mean():.4f} avg")
print(f"  → {'Parameters separable — sequential optimization valid.' if col_ranges.mean() > row_ranges.mean() else 'Interaction detected — consider joint search.'}")

# Conservative recommendation: use the largest min stable duration found
RECOMMENDED_NUM_TICKS = max(bisect_final.values()) // OPTIMAL_SPT
print("\n" + "=" * 60)
print("RECOMMENDED CONFIGURATION FOR OPTIMIZATION PIPELINE:")
print(f"  seconds_per_tick = {OPTIMAL_SPT}")
print(f"  num_ticks        = {RECOMMENDED_NUM_TICKS}  "
      f"({RECOMMENDED_NUM_TICKS * OPTIMAL_SPT // 60} min simulated)")
print("=" * 60)